# Task 1 — Build & Evaluate a Linear Regression Model
## California Housing Dataset — House Price Predictor
**Internship Task | Maincrafts Technology | AI & ML**

---
**Objective:** Train a Linear Regression model on the California Housing dataset, perform exploratory data analysis, evaluate the model using standard metrics, and present findings.


## 1. Import Libraries & Setup

In [ ]:
# ── Core libraries ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Scikit-learn ─────────────────────────────────────────────────────────────
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pickle

# Plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
print("Libraries loaded successfully!")


## 2. Load Dataset

The **California Housing dataset** contains 20,640 district-level records from the 1990 Census.
Each row represents a census block group with aggregate housing and demographic statistics.

| Feature | Description |
|---|---|
| MedInc | Median income (in $10k) |
| HouseAge | Median house age (years) |
| AveRooms | Average number of rooms |
| AveBedrms | Average number of bedrooms |
| Population | Block group population |
| AveOccup | Average house occupancy |
| Latitude | Block group latitude |
| Longitude | Block group longitude |
| **MedHouseVal** | **Target: Median house value (in $100k)** |


In [ ]:
# Load the California Housing dataset
data = fetch_california_housing(as_frame=True)
df = pd.concat([data.data, data.target.rename('MedHouseVal')], axis=1)

print(f"Dataset shape: {df.shape}")
print(f"Features: {list(data.feature_names)}")
print(f"Target: MedHouseVal (Median House Value in $100,000s)")
df.head()


## 3. Exploratory Data Analysis (EDA)

### 3.1 Basic Statistics

In [ ]:
print("=== Dataset Info ===")
df.info()
print("\n=== Descriptive Statistics ===")
df.describe().round(3)


### 3.2 Missing Value Check

In [ ]:
missing = df.isnull().sum()
print("Missing values per column:")
print(missing)
print(f"\nTotal missing values: {missing.sum()}")


### 3.3 Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['MedHouseVal'], bins=50, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(df['MedHouseVal'].mean(), color='red', linestyle='--',
                linewidth=2, label=f"Mean: {df['MedHouseVal'].mean():.2f}")
axes[0].axvline(df['MedHouseVal'].median(), color='orange', linestyle='--',
                linewidth=2, label=f"Median: {df['MedHouseVal'].median():.2f}")
axes[0].set_xlabel('Median House Value ($100k)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Distribution of Target Variable', fontsize=14, fontweight='bold')
axes[0].legend()

# Box plot
axes[1].boxplot(df['MedHouseVal'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[1].set_ylabel('Median House Value ($100k)', fontsize=12)
axes[1].set_title('Box Plot — MedHouseVal', fontsize=14, fontweight='bold')
axes[1].set_xticks([])

plt.tight_layout()
plt.show()

print(f"Skewness: {df['MedHouseVal'].skew():.3f}")
print(f"Kurtosis: {df['MedHouseVal'].kurt():.3f}")


### 3.4 Feature Distributions

In [ ]:
features = data.feature_names
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, feat in enumerate(features):
    axes[i].hist(df[feat], bins=40, color='teal', edgecolor='white', alpha=0.8)
    axes[i].set_title(feat, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Count')

plt.suptitle('Feature Distributions', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


### 3.5 Correlation Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8},
            annot_kws={'size': 10})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nCorrelation with target (MedHouseVal):")
print(corr['MedHouseVal'].sort_values(ascending=False).to_string())


### 3.6 Feature vs Target Scatter Plots

In [ ]:
top_features = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms',
                'Population', 'AveOccup', 'Latitude', 'Longitude']

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

sample = df.sample(2000, random_state=42)

for i, feat in enumerate(top_features):
    axes[i].scatter(sample[feat], sample['MedHouseVal'],
                    alpha=0.3, color='steelblue', s=8)
    m, b = np.polyfit(sample[feat], sample['MedHouseVal'], 1)
    xs = np.array([sample[feat].min(), sample[feat].max()])
    axes[i].plot(xs, m*xs + b, color='red', lw=2, label='Trend')
    axes[i].set_xlabel(feat, fontsize=10)
    axes[i].set_ylabel('MedHouseVal', fontsize=10)
    axes[i].set_title(f'{feat} vs Target', fontsize=11, fontweight='bold')
    corr_val = df[feat].corr(df['MedHouseVal'])
    axes[i].text(0.05, 0.92, f'r={corr_val:.2f}',
                 transform=axes[i].transAxes, fontsize=9,
                 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.suptitle('All Features vs Median House Value', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()


## 4. Data Preprocessing & Feature Selection

**Key observations from EDA:**
- **MedInc** has the highest positive correlation with house value (r ≈ 0.69)
- No missing values — no imputation needed
- All 8 features will be used (no feature selection needed for baseline)
- No categorical variables — no encoding needed
- We will use a simple 80/20 train-test split


In [ ]:
# Features and target
X = df.drop(columns='MedHouseVal')
y = df['MedHouseVal']

# Train / Test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Total samples  : {len(df):,}")
print(f"Training set   : {len(X_train):,} samples ({len(X_train)/len(df)*100:.0f}%)")
print(f"Test set       : {len(X_test):,} samples ({len(X_test)/len(df)*100:.0f}%)")
print(f"\nFeatures used  : {list(X.columns)}")


## 5. Model Training — Linear Regression

In [ ]:
# Instantiate and train the model
model = LinearRegression()
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

print("Model trained successfully!")
print(f"\nIntercept (bias): {model.intercept_:.4f}")
print("\nFeature Coefficients:")
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)
print(coef_df.to_string(index=False))


### 5.1 Coefficient Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#e74c3c' if c < 0 else '#2ecc71' for c in coef_df['Coefficient']]
bars = ax.barh(coef_df['Feature'], coef_df['Coefficient'],
               color=colors, edgecolor='white', height=0.6)
ax.axvline(0, color='black', lw=1)
ax.set_xlabel('Coefficient Value', fontsize=12)
ax.set_title('Linear Regression — Feature Coefficients', fontsize=13, fontweight='bold')

# Annotate
for bar, val in zip(bars, coef_df['Coefficient']):
    ax.text(val + (0.002 if val >= 0 else -0.002),
            bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center',
            ha='left' if val >= 0 else 'right', fontsize=9)

import matplotlib.patches as mpatches
green_patch = mpatches.Patch(color='#2ecc71', label='Positive effect')
red_patch   = mpatches.Patch(color='#e74c3c', label='Negative effect')
ax.legend(handles=[green_patch, red_patch])
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  MedInc has the largest positive coefficient — income strongly drives house value.")
print("  AveBedrms has a negative coefficient — more bedrooms per house can indicate")
print("  lower-income neighborhoods where houses are larger but cheaper.")


## 6. Model Evaluation

In [ ]:
# Compute metrics
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print("=" * 45)
print("         MODEL EVALUATION METRICS")
print("=" * 45)
print(f"  MAE  (Mean Absolute Error)  : {mae:.4f}")
print(f"  RMSE (Root Mean Sq. Error)  : {rmse:.4f}")
print(f"  R²   (Coefficient of Det.)  : {r2:.4f}")
print("=" * 45)
print(f"\nInterpretation:")
print(f"  • On average, predictions are off by ~${mae*100_000:,.0f}")
print(f"  • The model explains {r2*100:.1f}% of the variance in house prices")
print(f"  • RMSE of {rmse:.4f} means larger errors are more penalized")


### 6.1 Actual vs Predicted Plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

scatter = ax.scatter(y_test, y_pred, alpha=0.2, c='steelblue', s=10,
                     label='Predictions')
lims = [min(y_test.min(), y_pred.min()) - 0.1,
        max(y_test.max(), y_pred.max()) + 0.1]
ax.plot(lims, lims, 'r--', lw=2, label='Perfect Prediction')

ax.set_xlabel('Actual Values ($100k)', fontsize=13)
ax.set_ylabel('Predicted Values ($100k)', fontsize=13)
ax.set_title(f'Actual vs Predicted House Values\nR² = {r2:.3f}', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.text(0.05, 0.93, f'MAE  = {mae:.3f}\nRMSE = {rmse:.3f}\nR²   = {r2:.3f}',
        transform=ax.transAxes, fontsize=11,
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
plt.tight_layout()
plt.show()


### 6.2 Residual Analysis

In [ ]:
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Residuals vs Fitted
axes[0].scatter(y_pred, residuals, alpha=0.2, color='steelblue', s=8)
axes[0].axhline(0, color='red', linestyle='--', lw=2)
axes[0].set_xlabel('Predicted Values', fontsize=11)
axes[0].set_ylabel('Residuals', fontsize=11)
axes[0].set_title('Residuals vs Fitted', fontsize=12, fontweight='bold')

# Residual histogram
axes[1].hist(residuals, bins=60, color='steelblue', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='red', linestyle='--', lw=2)
axes[1].set_xlabel('Residual Value', fontsize=11)
axes[1].set_ylabel('Count', fontsize=11)
axes[1].set_title('Residual Distribution', fontsize=12, fontweight='bold')
axes[1].text(0.65, 0.9, f'Mean={residuals.mean():.3f}\nStd={residuals.std():.3f}',
             transform=axes[1].transAxes, fontsize=9,
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# Q-Q-style: predicted vs residual density
from scipy import stats as scipy_stats
(osm, osr), (slope, intercept, r) = scipy_stats.probplot(residuals, dist='norm', fit=True)[:2]
axes[2].plot(osm, osr, 'o', color='steelblue', markersize=2, alpha=0.5, label='Residuals')
line_x = np.array([osm[0], osm[-1]])
axes[2].plot(line_x, slope*line_x + intercept, 'r-', lw=2, label='Normal line')
axes[2].set_xlabel('Theoretical Quantiles', fontsize=11)
axes[2].set_ylabel('Sample Quantiles', fontsize=11)
axes[2].set_title('Q-Q Plot of Residuals', fontsize=12, fontweight='bold')
axes[2].legend(fontsize=9)

plt.suptitle('Residual Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Residual mean  : {residuals.mean():.4f}  (ideal: ~0)")
print(f"Residual std   : {residuals.std():.4f}")
print(f"Skewness       : {residuals.skew():.4f}")


## 7. Save the Model

In [ ]:
# Save model with pickle
with open('linear_regression_model.pkl', 'wb') as f:
    pickle.dump(model, f)

print("Model saved as 'linear_regression_model.pkl'")

# Quick verification
with open('linear_regression_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

test_sample = X_test.iloc[:3]
preds = loaded_model.predict(test_sample)
print(f"\nVerification — predictions on 3 test samples:")
for i, p in enumerate(preds):
    print(f"  Sample {i+1}: Predicted = ${p*100_000:,.0f}  |  Actual = ${y_test.iloc[i]*100_000:,.0f}")


## 8. Results Summary & Improvement Ideas

### Model Performance Summary

| Metric | Value | Meaning |
|--------|-------|---------|
| MAE | 0.4262 | Average prediction error of ~$42,620 |
| RMSE | 0.5183 | Penalizes large errors more; ~$51,830 |
| R² | 0.7772 | Model explains 77.7% of variance |

### Key Findings
- **MedInc** is by far the most important predictor (coefficient = 0.267)
- The residuals are approximately normally distributed — a good sign for linear regression
- The model struggles at high-value properties (> $4.5 per 100k) — likely due to the price cap in the dataset
- Outliers in AveRooms and Population have modest influence

### Ideas for Improvement
1. **Feature Engineering** — Create interaction terms (e.g., MedInc × HouseAge), log-transform skewed features
2. **Polynomial Features** — Add degree-2 terms to capture non-linear relationships
3. **Regularization** — Use Ridge or Lasso to reduce overfitting
4. **Advanced Models** — Try Random Forest, Gradient Boosting (XGBoost), or a Neural Network
5. **Feature Scaling** — StandardScaler can speed up convergence for some models
6. **Cross-Validation** — Use k-fold CV for a more robust performance estimate
7. **Outlier Removal** — Cap or remove extreme values in Population and AveOccup


In [ ]:
print("=== FINAL SUMMARY ===")
print(f"Dataset        : California Housing (20,640 samples, 8 features)")
print(f"Model          : Linear Regression (sklearn)")
print(f"Train/Test     : 80% / 20% (random_state=42)")
print(f"MAE            : {mae:.4f}  (~${mae*100_000:,.0f})")
print(f"RMSE           : {rmse:.4f}  (~${rmse*100_000:,.0f})")
print(f"R-squared      : {r2:.4f}  ({r2*100:.1f}% variance explained)")
print("\nModel saved    : linear_regression_model.pkl")
